<h3>실습 안내</h3><p>두 경우를 구분합니다. 첫째, 전체 문서에 답이 없습니다. 둘째, 정답 문서는 있지만 검색 결과에서 빠졌습니다. 모델의 답만 보지 않고 검색된 문서 ID도 함께 확인합니다.</p>

In [1]:
# 실제 임베딩 검색과 생성 호출에 필요한 패키지를 설치합니다.
# 버전을 ==로 고정해 반복 실행 환경을 일정하게 맞춥니다.
%pip install -q "langchain==1.4.0" "langchain-openai==1.6.0" "python-dotenv==1.2.3"

Note: you may need to restart the kernel to use updated packages.


<h3>1. API 키 준비</h3><p><code>.env</code>를 로드하고 외부 API를 사용할 수 있는지 확인합니다.</p>

In [2]:
# env_file은 현재 작업 폴더에서 찾은 .env 경로입니다.
# load_dotenv()는 파일을 로드하고 assert는 API 키 존재를 검사합니다.
import os
from dotenv import find_dotenv, load_dotenv

env_file = find_dotenv(usecwd=True)
assert env_file, ".env를 찾지 못했습니다."
load_dotenv(env_file)
assert os.getenv("OPENAI_API_KEY"), "OPENAI_API_KEY가 없습니다."
print("API 키 준비 완료")

API 키 준비 완료


<h3>2. 검색 실패를 보이는 문서 준비</h3><p>첫 문서 <code>DISTRACTOR</code>는 질문과 같은 단어가 많지만 답은 없습니다. 두 번째 문서 <code>ANSWER</code>에는 실제 연체 규정이 있습니다.</p>

In [3]:
# DISTRACTOR는 단어는 비슷하지만 답이 없고 ANSWER는 연체 규정을 담습니다.
# from_documents()는 두 Document를 실제로 임베딩한 store를 반환합니다.
from langchain_core.documents import Document
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_openai import OpenAIEmbeddings

documents = [
    Document(page_content="책을 3일 늦게 반납했습니다. 안내 데스크에 문의하세요.", metadata={"doc_id": "DISTRACTOR"}),
    Document(page_content="자료를 늦게 반납하면 연체 일수만큼 대출이 정지됩니다.", metadata={"doc_id": "ANSWER"}),
]
embedding = OpenAIEmbeddings(model="text-embedding-3-small")
store = InMemoryVectorStore.from_documents(documents, embedding)
print("문서 수:", len(documents))
for document in documents:
    print(document.metadata["doc_id"], document.page_content)

문서 수: 2
DISTRACTOR 책을 3일 늦게 반납했습니다. 안내 데스크에 문의하세요.
ANSWER 자료를 늦게 반납하면 연체 일수만큼 대출이 정지됩니다.


<h3>3. k=1 검색 중간값 관찰</h3><p>생성 모델을 호출하기 전에 어느 문서가 선택되었는지 확인합니다.</p>

In [4]:
# one_hit은 k=1로 가져온 Document 한 건의 리스트입니다.
# selected_id와 selected_text를 따로 출력해 검색 실패를 먼저 확인합니다.
question = "책을 3일 늦게 반납하면 어떤 불이익이 있나요?"
one_hit = store.similarity_search(question, k=1)
selected_id = one_hit[0].metadata["doc_id"]
selected_text = one_hit[0].page_content
print("k=1 ID:", selected_id)
print("k=1 본문:", selected_text)
assert selected_id == "DISTRACTOR"

k=1 ID: DISTRACTOR
k=1 본문: 책을 3일 늦게 반납했습니다. 안내 데스크에 문의하세요.


<h3>4. k=1 문맥으로 엄격한 답 생성</h3><p>검색 문서에 답이 없으면 “찾을 수 없습니다”라고만 답하도록 모델에 지시합니다.</p>

In [5]:
# f 삼중 문자열은 selected_text와 question을 여러 줄 프롬프트에 삽입합니다.
# missed_answer는 k=1 문맥만 본 모델의 AIMessage입니다.
from langchain.chat_models import init_chat_model

model = init_chat_model("openai:gpt-5.4-mini", max_tokens=80)
prompt_k1 = f"""문서에 답이 없으면 '찾을 수 없습니다'라고만 답하세요.

[문서]
{selected_text}

[질문]
{question}
"""
missed_answer = model.invoke(prompt_k1)
print(missed_answer.text)
assert "찾을 수 없습니다" in missed_answer.text

찾을 수 없습니다


<h3>5. k만 2로 바꾸어 검색 비교</h3><p>다른 값은 그대로 두고 <code>k</code>만 늘려 정답 문서가 포함되는지 확인합니다.</p>

In [6]:
# two_hits는 같은 question에서 k=2로 검색한 Document 리스트입니다.
# for로 ID와 본문을 따로 모아 정답 문서가 문맥에 추가되었는지 보여 줍니다.
two_hits = store.similarity_search(question, k=2)
two_ids = []
context_parts = []
for document in two_hits:
    two_ids.append(document.metadata["doc_id"])
    context_parts.append(document.page_content)
context_k2 = "\n".join(context_parts)
print("k=1:", [selected_id])
print("k=2:", two_ids)
print(context_k2)

k=1: ['DISTRACTOR']
k=2: ['DISTRACTOR', 'ANSWER']
책을 3일 늦게 반납했습니다. 안내 데스크에 문의하세요.
자료를 늦게 반납하면 연체 일수만큼 대출이 정지됩니다.


<h3>6. 수정된 문맥으로 다시 답하기</h3><p>같은 엄격한 지시와 질문을 유지하고 문맥만 <code>context_k2</code>로 바꿉니다.</p>

In [7]:
# prompt_k2와 prompt_k1의 핵심 차이는 검색 문맥입니다.
# recovered_answer.text에서 실제 연체 규정이 복구되었는지 확인합니다.
prompt_k2 = f"""문서에 답이 없으면 '찾을 수 없습니다'라고만 답하세요.

[문서]
{context_k2}

[질문]
{question}
"""
recovered_answer = model.invoke(prompt_k2)
print(recovered_answer.text)
assert "ANSWER" in two_ids
assert "대출" in recovered_answer.text
assert "정지" in recovered_answer.text

자료를 늦게 반납하면 연체 일수만큼 대출이 정지됩니다.


<h3>7. 문서 집합에 답이 없는 경우 비교</h3><p>RAG 문서 자체에 정보가 없으면 <code>k</code>를 늘려도 답을 만들 근거가 생기지 않습니다.</p>

In [8]:
# missing_context에는 A/S 처리 기간이 없어 정답 문서 자체가 존재하지 않습니다.
# missing_answer는 검색 누락이 아닌 문서 집합의 답 부재를 보여 줍니다.
missing_context = "책 대출과 반납 규정만 있으며 A/S 처리 기간은 없습니다."
missing_question = "고장 난 기기의 A/S 처리 기간은 며칠인가요?"
missing_prompt = f"""문서에 답이 없으면 '찾을 수 없습니다'라고만 답하세요.

[문서]
{missing_context}

[질문]
{missing_question}
"""
missing_answer = model.invoke(missing_prompt)
print(missing_answer.text)
assert "찾을 수 없습니다" in missing_answer.text

찾을 수 없습니다


<h3>정리</h3><p>모델이 동일하게 “찾을 수 없습니다”라고 답해도 원인은 다를 수 있습니다. 검색된 문서 ID를 보면 정답 문서가 검색에서 빠졌는지, 전체 문서에 답이 없는지 구분할 수 있습니다.</p>

<p><strong>원인</strong> — <code>k=1</code>이 DISTRACTOR만 가져와 정답 규정이 프롬프트에 들어가지 않았습니다.</p>

In [9]:
# before_ids와 missed_answer를 함께 출력해 검색 결과와 생성 결과를 연결합니다.
# ANSWER가 있어야 한다는 assert는 실패하고 except가 예상한 원인을 알립니다.
before_ids = [selected_id]
print("Before 검색:", before_ids)
print("Before 답:", missed_answer.text)
try:
    assert "ANSWER" in before_ids
except AssertionError:
    print("예상한 실패: 정답 문서가 문맥에 없습니다.")
else:
    raise AssertionError("검색 누락이 재현되어야 합니다.")

Before 검색: ['DISTRACTOR']
Before 답: 찾을 수 없습니다
예상한 실패: 정답 문서가 문맥에 없습니다.


<p><strong>수정</strong> — 앞에서 k를 2로 바꾸어 실행한 검색 ID와 생성된 답변을 다시 나란히 보여 줍니다. ANSWER가 포함되고 대출 정지 규정이 답에 있는지 검사하며, 이 셀에서는 모델을 추가로 호출하지 않습니다.</p>

In [10]:
# after_ids와 recovered_answer는 기본 예제에서 k=2로 실제 실행한 결과입니다.
# 정답 문서·대출·정지 세 조건을 나누어 검사합니다.
after_ids = two_ids
after_answer = recovered_answer.text
print("After 검색:", after_ids)
print("After 답:", after_answer)
assert "ANSWER" in after_ids
assert "대출" in after_answer
assert "정지" in after_answer

After 검색: ['DISTRACTOR', 'ANSWER']
After 답: 자료를 늦게 반납하면 연체 일수만큼 대출이 정지됩니다.
